# 📋 Vietnamese Fact-Checking: SER + TVC + RAG Pipeline

**🎯 Mục tiêu:** Xây dựng hệ thống xác thực thông tin tiếng Việt dựa trên kiến trúc 3 tầng:
1. **Semantic Evidence Retrieval (SER):** BM25 (Sparse) + BGE-M3 (Dense) + RRF Fusion + Cross-Encoder Reranking
2. **Two-step Verdict Classification (TVC):** Lọc tính đầy đủ (Sufficiency) → Xác minh lập trường (Stance)
3. **RAG Rationale Generation:** Sinh lời giải thích Chain-of-Thought qua LLM

**Datasets:**
- [High-Will/ViWikiFC](https://huggingface.co/datasets/High-Will/ViWikiFC) — Wikipedia tiếng Việt (20,919 mẫu)
- [tranthaihoa/vifactcheck](https://huggingface.co/datasets/tranthaihoa/vifactcheck) — Báo chí tiếng Việt đa lĩnh vực (7,232 mẫu)

**Đánh giá:** Hits@K, MRR@10, Macro-F1, Strict FEVER Score, Ablation Study

> **Lưu ý:** Notebook này được thiết kế để chạy trên **Google Colab** với GPU T4 hoặc A100.

---
## 0⃣ Cài đặt Môi trường (Environment Setup)
Chạy cell này đầu tiên để cài đặt toàn bộ thư viện cần thiết.

In [ ]:
# ============================================================
# 0. ENVIRONMENT SETUP - Install all required packages
# ============================================================
!pip install -q torch transformers datasets sentence-transformers \
    faiss-gpu rank-bm25 pyvi accelerate bitsandbytes \
    pydantic scikit-learn seaborn matplotlib tqdm

import warnings
warnings.filterwarnings('ignore')

import os
import re
import json
import time
import unicodedata
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field, asdict
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import faiss
from tqdm.auto import tqdm

from datasets import load_dataset, concatenate_datasets
from rank_bm25 import BM25Okapi
from pyvi.ViTokenizer import tokenize as vi_tokenize
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, pipeline
)
from pydantic import BaseModel, Field, ValidationError

from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\U0001f680 Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

---
## 1⃣ Module 1: Data Harmonization & Corpus Extraction

### Nhiệm vụ:
- Tải 2 bộ dữ liệu từ HuggingFace
- Chuẩn hóa schema chung: `{id, claim, evidence, label, source}`
- Chuẩn hóa nhãn: `SUPPORTED`, `REFUTED`, `NOT_ENOUGH_INFO`
- Chuẩn hóa văn bản tiếng Việt: Unicode NFC, tách từ
- Trích xuất corpus thống nhất cho Retrieval indexing

In [ ]:
# ============================================================
# 1. DATA HARMONIZATION & CORPUS EXTRACTION
# ============================================================

# --- 1.1 Canonical Data Schema ---
@dataclass
class FactCheckSample:
    """Canonical schema for a fact-checking sample."""
    id: str
    claim: str
    evidence: str
    label: str  # SUPPORTED | REFUTED | NOT_ENOUGH_INFO
    source: str  # 'viwikifc' | 'vifactcheck'


# --- 1.2 Label Normalization Maps ---
# ViWikiFC: string labels -> canonical
LABEL_MAP_VIWIKIFC = {
    'Supports': 'SUPPORTED',
    'supports': 'SUPPORTED',
    'SUPPORTS': 'SUPPORTED',
    'Refutes': 'REFUTED',
    'refutes': 'REFUTED',
    'REFUTES': 'REFUTED',
    'Not_Enough_Information': 'NOT_ENOUGH_INFO',
    'not_enough_information': 'NOT_ENOUGH_INFO',
    'NOT_ENOUGH_INFORMATION': 'NOT_ENOUGH_INFO',
    'NEI': 'NOT_ENOUGH_INFO',
}

# ViFactCheck: integer labels -> canonical
LABEL_MAP_VIFACTCHECK_INT = {
    0: 'SUPPORTED',
    1: 'REFUTED',
    2: 'NOT_ENOUGH_INFO',
}

# ViFactCheck: string labels (fallback)
LABEL_MAP_VIFACTCHECK_STR = {
    'Supported': 'SUPPORTED',
    'supported': 'SUPPORTED',
    'SUPPORTED': 'SUPPORTED',
    'Refuted': 'REFUTED',
    'refuted': 'REFUTED',
    'REFUTED': 'REFUTED',
    'Not Enough Information': 'NOT_ENOUGH_INFO',
    'not enough information': 'NOT_ENOUGH_INFO',
    'NEI': 'NOT_ENOUGH_INFO',
    'Not_Enough_Information': 'NOT_ENOUGH_INFO',
}

LABEL2ID = {'SUPPORTED': 0, 'REFUTED': 1, 'NOT_ENOUGH_INFO': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


# --- 1.3 Vietnamese Text Normalization ---
def normalize_vietnamese(text: str) -> str:
    """NFC normalization, strip HTML entities and control chars."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)       # HTML entities
    text = re.sub(r'<[^>]+>', ' ', text)             # HTML tags
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def segment_vietnamese(text: str) -> str:
    """Word-segment Vietnamese text for BM25."""
    try:
        return vi_tokenize(text)
    except Exception:
        return text


print('\u2705 Data schemas and text utilities defined.')

In [ ]:
# --- 1.4 Load and Harmonize Datasets ---

def load_viwikifc() -> List[FactCheckSample]:
    """Load High-Will/ViWikiFC and map to canonical schema."""
    print('\U0001f4e5 Loading High-Will/ViWikiFC...')
    ds = load_dataset('High-Will/ViWikiFC')
    samples = []
    for split_name in ds:
        for idx, row in enumerate(ds[split_name]):
            claim = normalize_vietnamese(row.get('claim', ''))
            evidence = normalize_vietnamese(
                row.get('evidence', '') or row.get('context', '')
            )
            raw_label = str(row.get('gold_label', '')).strip()
            label = LABEL_MAP_VIWIKIFC.get(raw_label)
            if not claim or not label:
                continue
            samples.append(FactCheckSample(
                id=f'viwikifc_{split_name}_{idx}',
                claim=claim,
                evidence=evidence,
                label=label,
                source='viwikifc',
            ))
    print(f'   \u2705 Loaded {len(samples)} samples from ViWikiFC')
    return samples


def load_vifactcheck() -> List[FactCheckSample]:
    """Load tranthaihoa/vifactcheck and map to canonical schema.
    Schema: Statement (claim), Evidence, Context, labels (0/1/2), Topic, Author, Url
    labels: 0=Supported, 1=Refuted, 2=NEI"""
    print('\U0001f4e5 Loading tranthaihoa/vifactcheck...')
    ds = load_dataset('tranthaihoa/vifactcheck')
    samples = []
    for split_name in ds:  # splits: train, dev, test
        for idx, row in enumerate(ds[split_name]):
            # ViFactCheck: 'Statement' is the claim column
            claim = normalize_vietnamese(
                row.get('Statement', '') or row.get('claim', '')
            )
            # 'Evidence' is the gold evidence
            evidence = normalize_vietnamese(
                row.get('Evidence', '') or row.get('Context', '')
            )
            # 'labels' is integer: 0=Supported, 1=Refuted, 2=NEI
            raw_label = row.get('labels')
            if isinstance(raw_label, int):
                label = LABEL_MAP_VIFACTCHECK_INT.get(raw_label)
            else:
                label = LABEL_MAP_VIFACTCHECK_STR.get(str(raw_label).strip())
            if not claim or not label:
                continue
            samples.append(FactCheckSample(
                id=f'vifactcheck_{split_name}_{idx}',
                claim=claim,
                evidence=evidence,
                label=label,
                source='vifactcheck',
            ))
    print(f'   \u2705 Loaded {len(samples)} samples from ViFactCheck')
    return samples


# --- Execute Loading ---
viwikifc_samples = load_viwikifc()
vifactcheck_samples = load_vifactcheck()
all_samples = viwikifc_samples + vifactcheck_samples

# Convert to DataFrame
df_all = pd.DataFrame([asdict(s) for s in all_samples])

print(f'\n\U0001f4ca Tổng số mẫu: {len(df_all)}')
print(f'\nPhân phối nhãn:')
print(df_all['label'].value_counts())
print(f'\nPhân phối nguồn:')
print(df_all['source'].value_counts())

In [ ]:
# --- 1.5 Build Unified Corpus for Retrieval ---

def build_corpus(df: pd.DataFrame) -> pd.DataFrame:
    """Extract unique non-empty evidence passages into a corpus."""
    corpus_texts = df['evidence'].dropna().unique().tolist()
    corpus_texts = [t for t in corpus_texts if len(t.strip()) > 10]
    corpus_df = pd.DataFrame({
        'corpus_id': [f'doc_{i}' for i in range(len(corpus_texts))],
        'text': corpus_texts,
    })
    # Pre-compute word-segmented version for BM25
    print(f'\U0001f50d Đang tách từ cho {len(corpus_df)} đoạn văn bản corpus...')
    corpus_df['text_segmented'] = [
        segment_vietnamese(t) for t in tqdm(corpus_df['text'], desc='Segmenting')
    ]
    return corpus_df

corpus_df = build_corpus(df_all)
print(f'\u2705 Corpus size: {len(corpus_df)} unique passages')
print(f'   Avg length: {corpus_df["text"].str.len().mean():.0f} chars')


# --- 1.6 Create Train/Test Splits ---
df_train, df_test = train_test_split(
    df_all, test_size=0.2, random_state=SEED, stratify=df_all['label']
)

# Subsample eval for Colab runtime (adjustable)
N_EVAL = 200
if len(df_test) > N_EVAL:
    df_eval = df_test.sample(n=N_EVAL, random_state=SEED, replace=False)
else:
    df_eval = df_test.copy()

df_train = df_train.reset_index(drop=True)
df_eval = df_eval.reset_index(drop=True)

print(f'\n\U0001f4ca Train: {len(df_train)} | Eval: {len(df_eval)}')
print(f'\nEval label distribution:')
print(df_eval['label'].value_counts())

---
## 2⃣ Module 2: Semantic Evidence Retrieval (SER) Engine

### Kiến trúc:
```
Claim → BM25 (Top-50) + Dense BGE-M3 (Top-50)
     → Reciprocal Rank Fusion (RRF, k=60)
     → Cross-Encoder Reranker (Top-30 → Top-3)
     → Retrieved Evidence Set
```

In [ ]:
# ============================================================
# 2. SEMANTIC EVIDENCE RETRIEVAL (SER) ENGINE
# ============================================================

class SparseRetriever:
    """BM25-based sparse retriever with Vietnamese word segmentation."""

    def __init__(self, corpus_df: pd.DataFrame):
        print('\U0001f4da Building BM25 sparse index...')
        self.corpus_df = corpus_df.reset_index(drop=True)
        tokenized_corpus = [
            doc.lower().split()
            for doc in tqdm(self.corpus_df['text_segmented'], desc='BM25 Indexing')
        ]
        self.bm25 = BM25Okapi(tokenized_corpus)
        print(f'   \u2705 BM25 index built with {len(tokenized_corpus)} documents.')

    def retrieve(self, query: str, top_k: int = 50) -> List[Tuple[int, float]]:
        """Return list of (doc_index, bm25_score) tuples."""
        segmented_query = segment_vietnamese(query).lower().split()
        scores = self.bm25.get_scores(segmented_query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(idx), float(scores[idx])) for idx in top_indices if scores[idx] > 0]


class DenseRetriever:
    """Dense retriever using sentence-transformers + FAISS."""

    def __init__(self, corpus_df: pd.DataFrame, model_name: str = 'BAAI/bge-m3'):
        print(f'\U0001f9e0 Loading dense encoder: {model_name}...')
        self.corpus_df = corpus_df.reset_index(drop=True)
        self.encoder = SentenceTransformer(model_name, device=str(device))

        # Encode corpus
        print(f'   Encoding {len(self.corpus_df)} corpus documents...')
        corpus_texts = self.corpus_df['text'].tolist()
        self.corpus_embeddings = self.encoder.encode(
            corpus_texts,
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        # Build FAISS index (Inner Product for cosine sim on normalized vectors)
        dim = self.corpus_embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(self.corpus_embeddings.astype('float32'))
        print(f'   \u2705 FAISS index built: {self.index.ntotal} vectors, dim={dim}')

    def retrieve(self, query: str, top_k: int = 50) -> List[Tuple[int, float]]:
        """Return list of (doc_index, cosine_score) tuples."""
        q_emb = self.encoder.encode(
            [query], normalize_embeddings=True, convert_to_numpy=True
        ).astype('float32')
        scores, indices = self.index.search(q_emb, top_k)
        return [(int(indices[0][i]), float(scores[0][i])) for i in range(top_k)]


def reciprocal_rank_fusion(
    results_list: List[List[Tuple[int, float]]],
    k: int = 60
) -> List[Tuple[int, float]]:
    """Reciprocal Rank Fusion (RRF) to combine multiple ranked lists."""
    rrf_scores = defaultdict(float)
    for results in results_list:
        for rank, (doc_idx, _) in enumerate(results, start=1):
            rrf_scores[doc_idx] += 1.0 / (k + rank)
    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return fused


print('\u2705 Retrieval classes defined.')

In [ ]:
# --- 2.2 Build Retrieval Indexes ---

sparse_retriever = SparseRetriever(corpus_df)
dense_retriever = DenseRetriever(corpus_df, model_name='BAAI/bge-m3')

In [ ]:
# --- 2.3 Cross-Encoder Reranker ---

class Reranker:
    """Cross-encoder reranker for final passage selection."""

    def __init__(self, model_name: str = 'BAAI/bge-reranker-v2-m3'):
        print(f'\U0001f3af Loading reranker: {model_name}...')
        self.cross_encoder = CrossEncoder(model_name, device=str(device))
        print('   \u2705 Reranker loaded.')

    def rerank(
        self, query: str, doc_indices: List[int],
        corpus_df: pd.DataFrame, top_k: int = 3
    ) -> List[Tuple[int, float]]:
        """Rerank candidates and return top-k (doc_index, score) tuples."""
        if not doc_indices:
            return []
        pairs = [
            [query, corpus_df.iloc[idx]['text']]
            for idx in doc_indices
        ]
        scores = self.cross_encoder.predict(pairs, show_progress_bar=False)
        scored = list(zip(doc_indices, scores.tolist()))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]


reranker = Reranker('BAAI/bge-reranker-v2-m3')

In [ ]:
# --- 2.4 Complete SER Pipeline ---

class SERPipeline:
    """Full Semantic Evidence Retrieval pipeline."""

    def __init__(self, sparse, dense, reranker, corpus_df,
                 sparse_top_k=50, dense_top_k=50,
                 fuse_top_k=30, final_top_k=3):
        self.sparse = sparse
        self.dense = dense
        self.reranker = reranker
        self.corpus_df = corpus_df
        self.sparse_top_k = sparse_top_k
        self.dense_top_k = dense_top_k
        self.fuse_top_k = fuse_top_k
        self.final_top_k = final_top_k

    def retrieve(self, claim: str) -> List[Dict]:
        """Full SER: Sparse + Dense -> RRF -> Reranker -> Top-K evidence."""
        # Step 1: Sparse retrieval (BM25)
        sparse_results = self.sparse.retrieve(claim, self.sparse_top_k)

        # Step 2: Dense retrieval (BGE-M3 + FAISS)
        dense_results = self.dense.retrieve(claim, self.dense_top_k)

        # Step 3: RRF Fusion
        fused = reciprocal_rank_fusion([sparse_results, dense_results], k=60)
        fused_indices = [idx for idx, _ in fused[:self.fuse_top_k]]

        # Step 4: Cross-encoder reranking
        reranked = self.reranker.rerank(
            claim, fused_indices, self.corpus_df, self.final_top_k
        )

        # Build output
        results = []
        for rank, (doc_idx, score) in enumerate(reranked, start=1):
            results.append({
                'rank': rank,
                'doc_idx': doc_idx,
                'text': self.corpus_df.iloc[doc_idx]['text'],
                'reranker_score': score,
            })
        return results


ser_pipeline = SERPipeline(
    sparse=sparse_retriever,
    dense=dense_retriever,
    reranker=reranker,
    corpus_df=corpus_df,
)

# Quick test
test_claim = df_eval.iloc[0]['claim']
print(f'\U0001f50e Test claim: {test_claim[:100]}...')
test_results = ser_pipeline.retrieve(test_claim)
for r in test_results:
    print(f"   Rank {r['rank']}: score={r['reranker_score']:.4f} | {r['text'][:80]}...")
print('\u2705 SER Pipeline ready.')

---
## 3⃣ Module 3: Two-step Verdict Classification (TVC)

### Quy trình 2 bước:
1. **Bước 1 — Sufficiency Filter:** LLM đánh giá bằng chứng có đủ thông tin để kiểm chứng hay không
   - `INSUFFICIENT` → Gán nhãn `NOT_ENOUGH_INFO` và bỏ qua Bước 2
   - `SUFFICIENT` → Tiếp tục Bước 2
2. **Bước 2 — Stance Verification:** LLM phân loại `SUPPORTED` hay `REFUTED`

In [ ]:
# ============================================================
# 3. TWO-STEP VERDICT CLASSIFICATION (TVC)
# ============================================================

# --- 3.1 Load LLM with 4-bit Quantization ---

LLM_MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

print(f'\U0001f916 Loading LLM: {LLM_MODEL_NAME} (4-bit quantized)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME, trust_remote_code=True
)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
llm_model.eval()

print(f'\u2705 LLM loaded successfully.')
print(f'   Model dtype: {llm_model.dtype}')
print(f'   GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# --- 3.2 LLM Generation Utility ---

def generate_llm_response(prompt: str, max_new_tokens: int = 256) -> str:
    """Generate text from the quantized LLM."""
    messages = [{'role': 'user', 'content': prompt}]
    text_input = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llm_tokenizer(
        text_input, return_tensors='pt', truncation=True, max_length=2048
    ).to(device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    # Decode only the generated portion
    response = llm_tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    return response


# Quick test
test_resp = generate_llm_response('Xin chào, bạn có thể giúp tôi kiểm chứng thông tin không?')
print(f'\U0001f4ac LLM test response: {test_resp[:200]}')

In [ ]:
# --- 3.3 Prompt Templates for TVC (Vietnamese few-shot) ---

SUFFICIENCY_PROMPT_TEMPLATE = '''Bạn là chuyên gia kiểm chứng thông tin tiếng Việt. Nhiệm vụ của bạn là đánh giá xem bằng chứng dưới đây có CHỨA ĐỦ thông tin để xác minh phát biểu hay không.

### Ví dụ:
Phát biểu: "Hà Nội là thủ đô của Việt Nam."
Bằng chứng: "Hà Nội là thủ đô của nước Cộng hòa xã hội chủ nghĩa Việt Nam, đồng thời là thành phố lớn thứ hai cả nước."
Kết luận: SUFFICIENT

Phát biểu: "Mặt Trăng có khí quyển dày đặc."
Bằng chứng: "Trái Đất quay quanh Mặt Trời với chu kỳ 365 ngày."
Kết luận: INSUFFICIENT

### Nhiệm vụ của bạn:
Phát biểu: "{claim}"
Bằng chứng: "{evidence}"

Chỉ trả lời MỘT từ duy nhất: SUFFICIENT hoặc INSUFFICIENT
Kết luận:'''


STANCE_PROMPT_TEMPLATE = '''Bạn là chuyên gia kiểm chứng thông tin tiếng Việt. Dựa trên bằng chứng được cung cấp, hãy xác định phát biểu được HỖ TRỢ hay BÁC BỎ bởi bằng chứng.

### Ví dụ:
Phát biểu: "Hà Nội là thủ đô của Việt Nam."
Bằng chứng: "Hà Nội là thủ đô của nước Cộng hòa xã hội chủ nghĩa Việt Nam."
Kết luận: SUPPORTED

Phát biểu: "Đất nước Việt Nam không giáp biển."
Bằng chứng: "Việt Nam có đường bờ biển dài 3,260 km."
Kết luận: REFUTED

### Nhiệm vụ của bạn:
Phát biểu: "{claim}"
Bằng chứng: "{evidence}"

Chỉ trả lời MỘT từ duy nhất: SUPPORTED hoặc REFUTED
Kết luận:'''


print('\u2705 TVC prompt templates defined.')

In [ ]:
# --- 3.4 TVC Classifier ---

class TVCClassifier:
    """Two-step Verdict Classification using LLM."""

    def __init__(self):
        self.sufficiency_prompt = SUFFICIENCY_PROMPT_TEMPLATE
        self.stance_prompt = STANCE_PROMPT_TEMPLATE

    def _parse_sufficiency(self, response: str) -> str:
        """Parse LLM response for sufficiency check."""
        response_upper = response.upper().strip()
        if 'INSUFFICIENT' in response_upper:
            return 'INSUFFICIENT'
        elif 'SUFFICIENT' in response_upper:
            return 'SUFFICIENT'
        # Fallback: check Vietnamese keywords
        lower = response.lower()
        if any(kw in lower for kw in ['không đủ', 'thiếu', 'không liên quan', 'không có']):
            return 'INSUFFICIENT'
        return 'SUFFICIENT'  # Default

    def _parse_stance(self, response: str) -> str:
        """Parse LLM response for stance classification."""
        response_upper = response.upper().strip()
        if 'REFUTED' in response_upper or 'REFUTE' in response_upper:
            return 'REFUTED'
        elif 'SUPPORTED' in response_upper or 'SUPPORT' in response_upper:
            return 'SUPPORTED'
        # Fallback Vietnamese
        lower = response.lower()
        if any(kw in lower for kw in ['bác bỏ', 'sai', 'không đúng', 'mâu thuẫn']):
            return 'REFUTED'
        return 'SUPPORTED'  # Default

    def classify(self, claim: str, evidence: str) -> Dict:
        """Two-step classification: Sufficiency -> Stance."""
        # Step 1: Sufficiency Check
        suff_prompt = self.sufficiency_prompt.format(
            claim=claim, evidence=evidence
        )
        suff_response = generate_llm_response(suff_prompt, max_new_tokens=32)
        sufficiency = self._parse_sufficiency(suff_response)

        if sufficiency == 'INSUFFICIENT':
            return {
                'verdict': 'NOT_ENOUGH_INFO',
                'step1_response': suff_response,
                'step2_response': None,
                'sufficiency': sufficiency,
            }

        # Step 2: Stance Verification
        stance_prompt = self.stance_prompt.format(
            claim=claim, evidence=evidence
        )
        stance_response = generate_llm_response(stance_prompt, max_new_tokens=32)
        verdict = self._parse_stance(stance_response)

        return {
            'verdict': verdict,
            'step1_response': suff_response,
            'step2_response': stance_response,
            'sufficiency': sufficiency,
        }


tvc_classifier = TVCClassifier()

# Quick test
test_evidence = df_eval.iloc[0]['evidence']
test_claim = df_eval.iloc[0]['claim']
test_result = tvc_classifier.classify(test_claim, test_evidence)
print(f'\U0001f9ea TVC Test:')
print(f'   Claim: {test_claim[:80]}...')
print(f'   Verdict: {test_result["verdict"]}')
print(f'   Sufficiency: {test_result["sufficiency"]}')
print(f'   Ground truth: {df_eval.iloc[0]["label"]}')

---
## 4⃣ Module 4: RAG Rationale Generation
Sinh lời giải thích Chain-of-Thought bằng tiếng Việt qua LLM, kèm structured output.

In [ ]:
# ============================================================
# 4. RAG RATIONALE GENERATION
# ============================================================

# --- 4.1 Pydantic Output Schema ---

class VerificationResult(BaseModel):
    """Structured output schema for fact verification."""
    verdict: str = Field(description='SUPPORTED | REFUTED | NOT_ENOUGH_INFO')
    confidence: float = Field(ge=0.0, le=1.0, default=0.5)
    key_evidence_spans: List[str] = Field(default_factory=list)
    reasoning_vi: str = Field(default='', description='Vietnamese CoT explanation')


# --- 4.2 Rationale Prompt ---

RATIONALE_PROMPT_TEMPLATE = '''Bạn là chuyên gia kiểm chứng thông tin tiếng Việt. Dựa trên bằng chứng, hãy giải thích lý do phán quyết.

Phát biểu: "{claim}"
Bằng chứng: "{evidence}"
Phán quyết: {verdict}

Trả lời dưới dạng JSON hợp lệ:
{{
  "verdict": "{verdict}",
  "confidence": <số từ 0.0 đến 1.0>,
  "key_evidence_spans": ["câu bằng chứng quan trọng 1", "câu 2"],
  "reasoning_vi": "Giải thích logic chi tiết bằng tiếng Việt"
}}

JSON:'''


def generate_rationale(
    claim: str, evidence: str, verdict: str
) -> VerificationResult:
    """Generate Chain-of-Thought rationale with structured output."""
    prompt = RATIONALE_PROMPT_TEMPLATE.format(
        claim=claim, evidence=evidence, verdict=verdict
    )
    raw_response = generate_llm_response(prompt, max_new_tokens=512)

    # Try parsing JSON from response
    try:
        json_match = re.search(r'\{[^{}]*\}', raw_response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            return VerificationResult(**parsed)
    except (json.JSONDecodeError, ValidationError):
        pass

    # Fallback: construct manually
    return VerificationResult(
        verdict=verdict,
        confidence=0.5,
        key_evidence_spans=[evidence[:200]] if evidence else [],
        reasoning_vi=raw_response[:500],
    )


print('\u2705 RAG Rationale module defined.')

---
## 5⃣ Module 5: Đánh giá Toàn diện (Formal Academic Evaluation)

### Các thước đo:
1. **SER Evaluation:** Hits@1, Hits@3, Hits@5, MRR@10
2. **TVC Evaluation:** Macro-F1, Accuracy, Per-class P/R/F1, Confusion Matrix
3. **Strict FEVER Score:** Đúng nhãn **VÀ** đúng bằng chứng
4. **Ablation Study:** So sánh 3 cấu hình (LLM thuần, RAG tiêu chuẩn, SER+TVC+RAG)

In [ ]:
# ============================================================
# 5. FORMAL ACADEMIC EVALUATION
# ============================================================

# --- 5.1 Retrieval Metrics ---

def compute_retrieval_metrics(
    gold_evidence: str,
    retrieved_docs: List[Dict],
    ks: List[int] = [1, 3, 5],
    mrr_cutoff: int = 10,
) -> Dict:
    """Compute Hits@K and MRR for a single query."""
    gold_norm = normalize_vietnamese(gold_evidence).lower().strip()
    if not gold_norm:
        return {f'hits@{k}': 0 for k in ks} | {'mrr': 0.0}

    ranks = []
    for i, doc in enumerate(retrieved_docs[:mrr_cutoff]):
        retrieved_norm = normalize_vietnamese(doc['text']).lower().strip()
        # Check substantial word overlap (at least 60% of gold in retrieved)
        gold_words = set(gold_norm.split())
        retrieved_words = set(retrieved_norm.split())
        if len(gold_words) > 0:
            overlap = len(gold_words & retrieved_words) / len(gold_words)
            if overlap >= 0.6:
                ranks.append(i + 1)

    results = {}
    for k in ks:
        results[f'hits@{k}'] = 1.0 if any(r <= k for r in ranks) else 0.0
    results['mrr'] = 1.0 / min(ranks) if ranks else 0.0
    return results


print('\u2705 Retrieval metrics defined.')

In [ ]:
# --- 5.2 Run Full Pipeline on Eval Set ---

def run_full_pipeline(
    df_eval: pd.DataFrame,
    ser_pipeline: SERPipeline,
    tvc_classifier: TVCClassifier,
    use_gold_evidence: bool = False,
    mode: str = 'SER_TVC_RAG',
) -> pd.DataFrame:
    """
    Run the full pipeline on evaluation set.
    Modes:
      - LLM_ONLY: No retrieval, LLM classifies claim directly (3-way)
      - RAG_SINGLE: Retrieval + single-step 3-way classification
      - SER_TVC_RAG: Full proposed method (Retrieval + Two-step TVC)
    """
    results = []
    print(f'\n\U0001f680 Running pipeline: {mode} | Eval size: {len(df_eval)}')

    for idx, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc=mode):
        claim = row['claim']
        gold_evidence = row['evidence']
        gold_label = row['label']

        record = {
            'id': row['id'],
            'claim': claim,
            'gold_label': gold_label,
            'gold_evidence': gold_evidence,
        }

        # === RETRIEVAL ===
        if mode == 'LLM_ONLY':
            retrieved_docs = []
            evidence_text = ''
        else:
            if use_gold_evidence:
                retrieved_docs = [{'rank': 1, 'text': gold_evidence, 'reranker_score': 1.0, 'doc_idx': -1}]
            else:
                retrieved_docs = ser_pipeline.retrieve(claim)
            evidence_text = ' '.join([d['text'] for d in retrieved_docs[:3]])

        # Retrieval metrics
        ret_metrics = compute_retrieval_metrics(gold_evidence, retrieved_docs)
        record.update(ret_metrics)

        # === CLASSIFICATION ===
        try:
            if mode == 'LLM_ONLY':
                direct_prompt = (
                    f'Phát biểu sau là đúng hay sai? '
                    f'Trả lời MỘT từ: SUPPORTED hoặc REFUTED hoặc NOT_ENOUGH_INFO\n'
                    f'Phát biểu: "{claim}"\nKết luận:'
                )
                resp = generate_llm_response(direct_prompt, max_new_tokens=32)
                resp_upper = resp.upper()
                if 'REFUTED' in resp_upper or 'REFUTE' in resp_upper:
                    pred_label = 'REFUTED'
                elif 'SUPPORTED' in resp_upper or 'SUPPORT' in resp_upper:
                    pred_label = 'SUPPORTED'
                else:
                    pred_label = 'NOT_ENOUGH_INFO'

            elif mode == 'RAG_SINGLE':
                single_prompt = (
                    f'Dựa trên bằng chứng, phán quyết phát biểu này.\n'
                    f'Trả lời MỘT từ: SUPPORTED hoặc REFUTED hoặc NOT_ENOUGH_INFO\n'
                    f'Phát biểu: "{claim}"\n'
                    f'Bằng chứng: "{evidence_text[:1000]}"\nKết luận:'
                )
                resp = generate_llm_response(single_prompt, max_new_tokens=32)
                resp_upper = resp.upper()
                if 'REFUTED' in resp_upper or 'REFUTE' in resp_upper:
                    pred_label = 'REFUTED'
                elif 'NOT_ENOUGH' in resp_upper or 'NEI' in resp_upper:
                    pred_label = 'NOT_ENOUGH_INFO'
                elif 'SUPPORTED' in resp_upper or 'SUPPORT' in resp_upper:
                    pred_label = 'SUPPORTED'
                else:
                    pred_label = 'NOT_ENOUGH_INFO'

            elif mode == 'SER_TVC_RAG':
                tvc_result = tvc_classifier.classify(claim, evidence_text[:1000])
                pred_label = tvc_result['verdict']

            else:
                pred_label = 'NOT_ENOUGH_INFO'

        except Exception as e:
            print(f'   \u26a0\ufe0f Error at index {idx}: {e}')
            pred_label = 'NOT_ENOUGH_INFO'

        record['pred_label'] = pred_label
        record['is_correct'] = int(pred_label == gold_label)

        # Strict FEVER Score: correct verdict AND gold evidence retrieved
        evidence_retrieved = ret_metrics.get('hits@3', 0) > 0
        record['strict_fever'] = int(
            pred_label == gold_label and evidence_retrieved
        ) if mode != 'LLM_ONLY' else 0

        results.append(record)

    return pd.DataFrame(results)


print('\u2705 Full pipeline evaluation function defined.')

In [ ]:
# --- 5.3 Execute Evaluation: SER + TVC + RAG (Proposed Method) ---

results_ser_tvc = run_full_pipeline(
    df_eval, ser_pipeline, tvc_classifier, mode='SER_TVC_RAG'
)

In [ ]:
# --- 5.4 Execute Ablation: LLM Only (No RAG) ---

results_llm_only = run_full_pipeline(
    df_eval, ser_pipeline, tvc_classifier, mode='LLM_ONLY'
)

In [ ]:
# --- 5.5 Execute Ablation: RAG Single-step 3-way ---

results_rag_single = run_full_pipeline(
    df_eval, ser_pipeline, tvc_classifier, mode='RAG_SINGLE'
)

---
## 6⃣ Kết quả Đánh giá & Ablation Study

In [ ]:
# ============================================================
# 6. RESULTS VISUALIZATION & ABLATION STUDY
# ============================================================

def print_full_report(results_df: pd.DataFrame, mode_name: str):
    """Print comprehensive evaluation report."""
    labels_list = ['SUPPORTED', 'REFUTED', 'NOT_ENOUGH_INFO']

    y_true = results_df['gold_label'].tolist()
    y_pred = results_df['pred_label'].tolist()

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=labels_list, average='macro', zero_division=0)

    print(f'\n{"=" * 60}')
    print(f'  \U0001f4cb EVALUATION REPORT: {mode_name}')
    print(f'{"=" * 60}')

    # Retrieval Metrics
    if 'hits@1' in results_df.columns:
        print(f'\n\U0001f50d Retrieval Metrics (SER):')
        for k in [1, 3, 5]:
            col = f'hits@{k}'
            if col in results_df.columns:
                print(f'   Hits@{k}: {results_df[col].mean():.4f}')
        if 'mrr' in results_df.columns:
            print(f'   MRR@10:  {results_df["mrr"].mean():.4f}')

    # Classification Metrics
    print(f'\n\U0001f3af Classification Metrics (TVC):')
    print(f'   Accuracy:  {acc:.4f}')
    print(f'   Macro-F1:  {macro_f1:.4f}')

    # Per-class report
    print(f'\n\U0001f4ca Per-class Report:')
    print(classification_report(
        y_true, y_pred, labels=labels_list, target_names=labels_list, zero_division=0
    ))

    # Strict FEVER Score
    if 'strict_fever' in results_df.columns:
        fever_score = results_df['strict_fever'].mean()
        print(f'\U0001f525 Strict FEVER Score: {fever_score:.4f}')

    return {'mode': mode_name, 'accuracy': acc, 'macro_f1': macro_f1,
            'fever_score': results_df.get('strict_fever', pd.Series([0])).mean()}


# Print reports
report_proposed = print_full_report(results_ser_tvc, 'SER + TVC + RAG (Proposed)')
report_llm = print_full_report(results_llm_only, 'LLM Only (No RAG)')
report_rag = print_full_report(results_rag_single, 'RAG Single-step 3-way')

In [ ]:
# --- 6.2 Ablation Study Comparison Table ---

ablation_data = pd.DataFrame([
    report_llm, report_rag, report_proposed
])

print('\n' + '=' * 60)
print('  \U0001f52c ABLATION STUDY COMPARISON')
print('=' * 60)
print(ablation_data.to_string(index=False, float_format='%.4f'))

# Highlight best
best_f1_mode = ablation_data.loc[ablation_data['macro_f1'].idxmax(), 'mode']
print(f'\n\u2b50 Best Macro-F1: {best_f1_mode}')

In [ ]:
# --- 6.3 Confusion Matrix (Proposed Method) ---

labels_list = ['SUPPORTED', 'REFUTED', 'NOT_ENOUGH_INFO']

cm = confusion_matrix(
    results_ser_tvc['gold_label'],
    results_ser_tvc['pred_label'],
    labels=labels_list
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels_list,
    yticklabels=labels_list,
    ax=ax,
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix: SER + TVC + RAG (Proposed Method)', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix_proposed.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Confusion matrix saved.')

In [ ]:
# --- 6.4 Ablation Bar Chart ---

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['accuracy', 'macro_f1', 'fever_score']
titles = ['Accuracy', 'Macro-F1', 'Strict FEVER Score']
colors = ['#4C72B0', '#DD8452', '#55A868']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    bars = ax.bar(
        ablation_data['mode'].str.replace(' ', '\n'),
        ablation_data[metric],
        color=color, alpha=0.85, edgecolor='black', linewidth=0.5
    )
    for bar, val in zip(bars, ablation_data[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Ablation Study: So sánh 3 cấu hình', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Ablation study chart saved.')

---
## 7⃣ Lưu Kết quả & Model Artifacts

In [ ]:
# ============================================================
# 7. SAVE RESULTS & MODEL ARTIFACTS
# ============================================================

OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save evaluation results
results_ser_tvc.to_csv(f'{OUTPUT_DIR}/results_ser_tvc_rag.csv', index=False)
results_llm_only.to_csv(f'{OUTPUT_DIR}/results_llm_only.csv', index=False)
results_rag_single.to_csv(f'{OUTPUT_DIR}/results_rag_single.csv', index=False)
ablation_data.to_csv(f'{OUTPUT_DIR}/ablation_study.csv', index=False)

# Save corpus and eval splits
corpus_df.to_parquet(f'{OUTPUT_DIR}/unified_corpus.parquet', index=False)
df_train.to_parquet(f'{OUTPUT_DIR}/train_split.parquet', index=False)
df_eval.to_parquet(f'{OUTPUT_DIR}/eval_split.parquet', index=False)

# Save sample rationale outputs
print('\n\U0001f4be Generating sample rationale outputs...')
sample_rationales = []
for idx in range(min(5, len(df_eval))):
    row = df_eval.iloc[idx]
    pred_label = results_ser_tvc.iloc[idx]['pred_label']
    try:
        rationale = generate_rationale(
            row['claim'], row['evidence'], pred_label
        )
        sample_rationales.append(rationale.model_dump())
        print(f'   Sample {idx+1}: {rationale.verdict} | {rationale.reasoning_vi[:80]}...')
    except Exception as e:
        print(f'   Sample {idx+1}: Error - {e}')

with open(f'{OUTPUT_DIR}/sample_rationales.json', 'w', encoding='utf-8') as f:
    json.dump(sample_rationales, f, ensure_ascii=False, indent=2)

print(f'\n\u2705 All results saved to {OUTPUT_DIR}/')
print('\nSaved files:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'   {fname}: {size_kb:.1f} KB')

---
## ✅ Hoàn tất!

### Tóm tắt kết quả:
- **Retrieval (SER):** Hits@K và MRR đánh giá chất lượng module tìm kiếm bằng chứng
- **Classification (TVC):** Macro-F1 và Accuracy đánh giá khả năng phân loại 3 nhãn
- **Strict FEVER Score:** Thước đo khắt khe nhất, yêu cầu đúng cả nhãn lẫn bằng chứng
- **Ablation Study:** Chứng minh SER+TVC+RAG vượt trội hơn LLM thuần và RAG tiêu chuẩn

### Output files:
| File | Mô tả |
|------|--------|
| `outputs/results_ser_tvc_rag.csv` | Kết quả phương pháp đề xuất |
| `outputs/results_llm_only.csv` | Kết quả LLM thuần (không RAG) |
| `outputs/results_rag_single.csv` | Kết quả RAG tiêu chuẩn |
| `outputs/ablation_study.csv` | Bảng so sánh Ablation Study |
| `outputs/sample_rationales.json` | Ví dụ lời giải thích CoT |
| `outputs/unified_corpus.parquet` | Corpus thống nhất |
| `confusion_matrix_proposed.png` | Ma trận nhầm lẫn |
| `ablation_study.png` | Biểu đồ so sánh Ablation |

### Nguồn dữ liệu:
- [High-Will/ViWikiFC](https://huggingface.co/datasets/High-Will/ViWikiFC) — ViWikiFC: Wikipedia-based Vietnamese Fact-Checking
- [tranthaihoa/vifactcheck](https://huggingface.co/datasets/tranthaihoa/vifactcheck) — ViFactCheck: Multi-domain News Fact-Checking (AAAI 2025)

### Mô hình sử dụng:
- **Dense Retriever:** [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3)
- **Cross-Encoder Reranker:** [BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3)
- **LLM (TVC + RAG):** [Qwen/Qwen2.5-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct) (4-bit NF4)